# Data Cleaning 

This notebook cleans the raw data available in data/raw and writes the clean version back to the folder data/processed. 

In [176]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np 
from c08_farming_exit import config, features, data_cleaning, mappings

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [54]:
# #In case you want to run Stata in a cell using the magic command %%stata, initialize it first!
# from c08_farming_exit.stata_utils import init_stata
# init_stata()

In [113]:
COUNTRIES = {
    "Botswana": config.RAW_DATA_DIR / "Botswana",
    "Kenya":    config.RAW_DATA_DIR / "Kenya",
    "Namibia":  config.RAW_DATA_DIR / "Namibia",
    "Tanzania": config.RAW_DATA_DIR / "Tanzania",
    "Zambia":   config.RAW_DATA_DIR / "Zambia",
}

## 1. Database

In [112]:
#THE DATABASE CONSISTS OF ADULTS ONLY
database = []

for country, base_path in COUNTRIES.items():
    identifying_info    = data_cleaning.load_and_preprocess(base_path, f"{country}_identifying_info.csv",                  features.IDENTIFYING_INFO_2023)
    
    hh_members          = data_cleaning.load_and_preprocess(base_path, f"{country}_household_members_characterstics.csv",  features.HH_MEMBERS_2023)
    hh_members          = data_cleaning.add_years_of_schooling(hh_members, mappings.education_mapping)

    merge = identifying_info.merge(hh_members, on=["interview_key"], how="inner")
    
    filtered = merge[
        merge["relation_to_head"].isin([    "Self/Head", 
                                            "Wife/Husband", 
                                            "Son/Daughter-In-Law", 
                                            "Sister/Brother", 
                                            "Mother/Father", 
                                            "Brother/Sister-In-Law", 
                                            "Grandfather/Mother", 
                                            "Father/Mother-In-Law"]) &
                                        (merge["age"] >= 18)
    ]

    database.append(filtered)

df_database = pd.concat(database, ignore_index=True)

#CREATE PERSONAL IDENTIFIER 
df_database["personal_id"] = df_database["country"] + "_" + df_database["interview_key"].astype(str) + "_" + df_database["members_id"].astype(str)

#FINAL SORTING
df_database = df_database[['country', 'region', 'district', 'enumeration_area', 'personal_id', 'interview_key', 'relation_to_head', 'gender', 'age', 'years_of_schooling', 'education_level']] \
                .query("country != 'YES+A112:L126+A112:C126'")

[Botswana_identifying_info.csv] Missing columns: ['ea', 'region']
[Namibia_identifying_info.csv] Missing columns: ['dist']


## 2. Features

### 2.1 Creating HH-Level Features

In [177]:
hh_features = []

for country, base_path in COUNTRIES.items():
    #NO CLEANING NECESSARY - one observation per hh
    # land_ownership          = data_cleaning.load_and_preprocess(base_path, f"{country}_land_ownership_and_access.csv",         features.LAND_OWNERSHIP_ACCESS_2023)
    # land_ownership          = data_cleaning.convert_land_sizes_to_acres(land_ownership, country)
    # crop_expenditure        = data_cleaning.load_and_preprocess(base_path, f"{country}_expenditure_on_crops.csv",              features.CROP_EXPENDITURE_2023)
    # lifestock_grazing       = data_cleaning.load_and_preprocess(base_path, f"{country}_grazing_patterns_and_schemes.csv",      features.LIFESTOCK_GRAZING_2023)
    # livestock_income        = data_cleaning.load_and_preprocess(base_path, f"{country}_income_livestock.csv",                  features.LIFESTOCK_INCOME_2023)
    # livestock_expenditure   = data_cleaning.load_and_preprocess(base_path, f"{country}_expenditure_livestock.csv",             features.LIFESTOCK_EXPENDITURE_2023)
    # housing_conditions      = data_cleaning.load_and_preprocess(base_path, f"{country}_housing_conditions.csv",                features.HOUSING_CONDITIONS_2023)
    # energy_access           = data_cleaning.load_and_preprocess(base_path, f"{country}_access_to_energy.csv",                  features.ENERGY_ACCESS_2023)
    # internet_access         = data_cleaning.load_and_preprocess(base_path, f"{country}_internet_access.csv",                   features.INTERNET_ACCESS_2023)
    # social_network          = data_cleaning.load_and_preprocess(base_path, f"{country}_other_household_social_network.csv",    features.SOCIAL_NETWORK_2023)
    # social_embeddedness     = data_cleaning.load_and_preprocess(base_path, f"{country}_social_embeddedness.csv",               features.SOCIAL_EMBEDDEDNESS_2023)
    # food_insecurity         = data_cleaning.load_and_preprocess(base_path, f"{country}_food_insecurity_experiance_scale.csv",  features.FOOD_INSECURITY_2023)
    # road_connectivity       = data_cleaning.load_and_preprocess(base_path, f"{country}_road_connectivity.csv",                 features.ROAD_CONNECTIVITY_2023)

    # #CLEANING NECESSARY - many observations per hh
    # market_access           = data_cleaning.load_and_preprocess(base_path, f"{country}_market_access.csv",                     features.MARKET_ACCESS_2023)
    # #market data is missing for Tanzania!
    # if market_access is None:
    #     print(f"  -> missing market_access file for {country}")
    #     continue
    # market_access           = data_cleaning.resolve_duplicates(market_access, key_col="interview_key", sort_col="crop_contract_crop_type", ascending=True)
    # market_access           = data_cleaning.add_missing_indicators(market_access, ["market_output_distance_in_km", "market_input_distance_in_km"], sentinel=99999, suffix='_missing')
    crop_production         = data_cleaning.load_and_preprocess(base_path, f"{country}_crop_production.csv",                   features.CROP_PRODUCTION_2023) #different crops
    # livestock_ownership     = data_cleaning.load_and_preprocess(base_path, f"{country}_livestock_ownership.csv",               features.LIVESTOCK_OWNERSHIP_2023) #different animals
    # assets_owned            = data_cleaning.load_and_preprocess(base_path, f"{country}_assets.csv",                            features.ASSETS_OWNED_2023)
    # shocks_and_coping       = data_cleaning.load_and_preprocess(base_path, f"{country}_shocks_and_coping.csv",                 features.SHOCKS_AND_COPING_2023)
    # other_income            = data_cleaning.load_and_preprocess(base_path, f"{country}_other_income.csv",                      features.OTHER_INCOME_SOURCES_2023)


    #SOME TABLES ARE NOT AVAILABLE FOR EACH COUNTRY: optional_merges solves this as it only merges available tables
    optional_merges = [
        # (land_ownership,          ["interview_key"],     "outer"),
        # (crop_expenditure,        ["interview_key"],     "outer"),
        # (lifestock_grazing,       ["interview_key"],     "outer")
        # (livestock_income,        ["interview_key"],     "outer")
        # (livestock_expenditure,   ["interview_key"],     "outer")
        # (housing_conditions,      ["interview_key"],     "outer")
        # (energy_access,           ["interview_key"],     "outer")
        # (internet_access,         ["interview_key"],     "outer")
        # (social_network,          ["interview_key"],     "outer")
        # (social_embeddedness,     ["interview_key"],     "outer")
        # (food_insecurity,         ["interview_key"],     "outer")
        # (road_connectivity,       ["interview_key"],     "outer")
        # (market_access,           ["interview_key"],     "outer")
        (crop_production,    ["interview_key"],     "outer")


    ]

    df_help = None

    for df, keys, how in optional_merges:
        if df is not None:
            if df_help is None:
                df_help = df
            else:
                df_help = df_help.merge(df, on=keys, how=how)
                
    if df_help is not None:            
        #Adding the country to the table for identification
        df_help.insert(0, "country", country)

    hh_features.append(df_help)


df_hh_features = pd.concat(hh_features, ignore_index=True)



In [178]:
missing_by_country = (
    df_hh_features
    .groupby('country')
    .apply(lambda g: g.isna().mean())
    .sort_index()
)
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(missing_by_country)


C:\Users\localuser\AppData\Local\Temp\ipykernel_16092\763753577.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.isna().mean())


,country,interview_key,crop_type,crop_harvested,crop_output,crop_unit,crop_sale,crop_sale_amount,crop_sale_unit,crop_sale_price_per_unit,crop_buyer_market,crop_buyer_trader,crop_buyer_cooperative,crop_buyer_commercial_farm,crop_buyer_hospitality,crop_buyer_government,crop_home_consumption,crop_home_consumption_unit,crop_storage,crop_storage_amount,crop_storage_unit,crop_organic_fertilizer,crop_inorganic_fertilizer,crop_pesticides,crop_tractor
country,,,,,,,,,,,,,,,,,,,,,,,,,
Botswana,0.0,0.0,0.0,0.0,0.615702,0.619835,0.619835,0.925620,0.925620,0.925620,0.929752,0.925620,0.925620,0.925620,0.925620,0.925620,0.619835,0.657025,0.0,0.880165,0.880165,0.049587,0.049587,0.049587,0.049587
Kenya,0.0,0.0,0.0,0.0,0.712305,0.714038,0.714038,0.837088,0.838821,0.838821,0.842288,0.844021,0.840555,0.838821,0.840555,0.838821,0.714038,0.741768,0.0,0.911612,0.911612,0.000000,0.000000,0.000000,0.000000
Namibia,0.0,0.0,0.0,0.0,0.398361,0.429508,0.429508,0.914754,0.914754,0.914754,0.916393,0.914754,0.916393,0.914754,0.914754,0.914754,0.429508,0.595082,0.0,0.640984,0.645902,0.027869,0.027869,0.027869,0.027869
Tanzania,0.0,0.0,0.0,0.0,0.174949,0.186521,0.186521,0.608577,0.617427,0.617427,0.622873,0.625596,0.617427,0.617427,0.617427,0.617427,0.186521,0.335602,0.0,0.513955,0.520762,0.002042,0.002042,0.002042,0.002042
Zambia,0.0,0.0,0.0,0.0,0.339400,0.370450,0.370450,0.936831,0.937901,0.937901,0.937901,0.941113,0.937901,0.937901,0.938972,0.937901,0.370450,0.432548,0.0,0.516060,0.521413,0.011777,0.011777,0.011777,0.011777


In [183]:
# df_hh_features["crop_contract"].value_counts()
df_hh_features.groupby("country")["crop_unit"].value_counts()

country   crop_unit    
Botswana  50 kg bag        29
          Numbers          16
          Kilograms        12
          10kg bag          6
          Bale / Bundle     6
                           ..
Zambia    Basins            4
          Crates            2
          Bunches           1
          Grams             1
          Liters            1
Name: count, Length: 67, dtype: int64

In [182]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(df_hh_features[df_hh_features["country"]=="Botswana"])

,country,interview_key,crop_type,crop_harvested,crop_output,crop_unit,crop_sale,crop_sale_amount,crop_sale_unit,crop_sale_price_per_unit,crop_buyer_market,crop_buyer_trader,crop_buyer_cooperative,crop_buyer_commercial_farm,crop_buyer_hospitality,crop_buyer_government,crop_home_consumption,crop_home_consumption_unit,crop_storage,crop_storage_amount,crop_storage_unit,crop_organic_fertilizer,crop_inorganic_fertilizer,crop_pesticides,crop_tractor
0,Botswana,95-91-52-45,MAIZE,Yes,25.0,10kg bag,Yes,20.00,10kg bag,80.0,No,No,Yes,No,No,No,5.0,10kg bag,No,NaN,NaN,No,Yes,Yes,No
1,Botswana,95-91-52-45,MILLET/MAIWA,Yes,25.0,10kg bag,Yes,20.00,10kg bag,80.0,No,No,Yes,No,No,No,5.0,10kg bag,No,NaN,NaN,NaN,NaN,NaN,NaN
2,Botswana,93-61-95-87,ONION,Yes,2.5,2kg packet,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,2kg packet,No,NaN,NaN,Yes,No,No,No
3,Botswana,93-61-95-87,IRISH POTATO,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN,Yes,No,No,No
4,Botswana,93-61-95-87,PUMPKIN,Yes,5.0,Kilograms,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,Kilograms,No,NaN,NaN,Yes,No,No,No
5,Botswana,93-61-95-87,CHILLI PEPPER (SHOMBO),Yes,500.0,Grams,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,500.0,Grams,No,NaN,NaN,Yes,No,No,No
6,Botswana,71-09-05-63,GUAVA,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN,NaN,NaN,NaN,NaN
7,Botswana,71-09-05-63,MANGO,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN,NaN,NaN,NaN,NaN
8,Botswana,81-05-32-56,BEANS/COWPEA,Yes,20.0,Liters,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,Liters,Yes,0.5,Other (please specify),NaN,NaN,NaN,NaN
9,Botswana,81-05-32-56,MAIZE,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN,NaN,NaN,NaN,NaN


### 2.2 Creating Individual-Level Features

In [62]:
individual_features = []

for country, base_path in COUNTRIES.items():
    #NO CLEANING NECESSARY 

    #CLEANING NECESSARY


    #SOME FEATURES ARE NOT AVAILABLE FOR EACH COUNTRY: optional_merges solves this as it only merges available features
    optional_merges = [
    ]

    df_help = None

    for df, keys, how in optional_merges:
        if df is not None:
            if df_help is None:
                df_help = df
            else:
                df_help = df_help.merge(df, on=keys, how=how)

    individual_features.append(df_help)

df_individual_features = pd.concat(individual_features, ignore_index=True)


ValueError: All objects passed were None

## 3. Write clean data to data/raw folder

In [ ]:
df.to_csv(config.PROCESSED_DATA_DIR / "clean_data.csv", index=False)